# Notebook 66 - LSTM Improvement Experiments

Bu notebookun amacı, mevcut en iyi LSTM modelini (Notebook 61-62) sistematik olarak geliştirmektir.

Başlangıç (Baseline):
- Leave-One-Group-Out (LOGO)
- StandardScaler
- Gradient Clipping
- Weighted CrossEntropy
- 12 saniye pencere (120 timestep)

Denenecek geliştirmeler:

1. Bidirectional LSTM
2. Trip-wise Normalization
3. Window Size Analysis
4. Missing Value Indicator Features
5. Dropout Optimization

Her deney bağımsız olarak değerlendirilecek ve baseline ile karşılaştırılacaktır.

In [1]:
import pandas as pd

results = pd.DataFrame(columns=[
    "Experiment",
    "Accuracy",
    "Macro F1",
    "Notes"
])

results

,Experiment,Accuracy,Macro F1,Notes


In [2]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path

import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [3]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

DATASET_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "uah_dataset.npz"
)

sensor_data = np.load(DATASET_PATH)

X = sensor_data["X"]
y = sensor_data["y"]
groups = sensor_data["groups"]

print(X.shape)

(30676, 120, 13)


In [5]:
unique_groups = np.unique(groups)

trip_labels = np.array([
    y[groups == trip][0]
    for trip in unique_groups
])

train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
    stratify=trip_labels,
)

In [6]:
train_mask = np.isin(
    groups,
    train_groups,
)

test_mask = np.isin(
    groups,
    test_groups,
)

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

In [7]:
print(X_train.shape)
print(X_test.shape)

(24725, 120, 13)
(5951, 120, 13)


In [8]:
from sklearn.preprocessing import StandardScaler

# (samples, timesteps, features)
n_train, seq_len, n_features = X_train.shape
n_test = X_test.shape[0]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(n_train, seq_len, n_features)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(n_test, seq_len, n_features)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(24725, 120, 13)
(5951, 120, 13)


In [9]:
from torch.utils.data import Dataset

class DriverDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [10]:
from torch.utils.data import DataLoader

batch_size = 32

train_dataset = DriverDataset(X_train_scaled, y_train)
test_dataset = DriverDataset(X_test_scaled, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [11]:
import torch.nn as nn

class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=13,
        hidden_size=64,
        num_layers=2,
        num_classes=3,
        dropout=0.3
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        x = hidden[-1]

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [13]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print(class_weights)

tensor([0.8212, 0.9869, 1.3004])


In [14]:
model = LSTMClassifier().to(device)

In [15]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [17]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [18]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    loss = running_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return loss, acc, macro_f1, all_labels, all_preds

In [33]:
import copy

num_epochs = 20
patience = 3

best_model = None
best_f1 = 0.0
patience_counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1, _, _ = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    test_f1s.append(test_f1)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f} | "
        f"Macro F1: {test_f1:.4f}"
    )

    if test_f1 > best_f1:

        best_f1 = test_f1
        patience_counter = 0

        best_model = copy.deepcopy(model.state_dict())


    else:

        patience_counter += 1

    if patience_counter >= patience:

        print("\nEarly stopping!")
        break

Epoch 01/20 | Train Loss: 0.2441 | Train Acc: 0.9045 | Test Loss: 1.2639 | Test Acc: 0.6175 | Macro F1: 0.6308
Epoch 02/20 | Train Loss: 0.2317 | Train Acc: 0.9096 | Test Loss: 1.1865 | Test Acc: 0.6431 | Macro F1: 0.6475
Epoch 03/20 | Train Loss: 0.2159 | Train Acc: 0.9148 | Test Loss: 1.1109 | Test Acc: 0.6612 | Macro F1: 0.6659
Epoch 04/20 | Train Loss: 0.2105 | Train Acc: 0.9160 | Test Loss: 1.3435 | Test Acc: 0.6224 | Macro F1: 0.6349
Epoch 05/20 | Train Loss: 0.2066 | Train Acc: 0.9184 | Test Loss: 1.1716 | Test Acc: 0.6619 | Macro F1: 0.6645
Epoch 06/20 | Train Loss: 0.1945 | Train Acc: 0.9237 | Test Loss: 1.3213 | Test Acc: 0.6641 | Macro F1: 0.6650

Early stopping!


In [34]:
model.load_state_dict(best_model)

<All keys matched successfully>

In [35]:
test_loss, test_acc, test_f1, y_true, y_pred = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"\nFinal Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")


Final Accuracy : 0.6612
Final Macro F1 : 0.6659


In [36]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)

print(cm)

              precision    recall  f1-score   support

           0       0.72      0.58      0.64      2955
           1       0.51      0.62      0.56      1495
           2       0.74      0.86      0.79      1501

    accuracy                           0.66      5951
   macro avg       0.66      0.69      0.67      5951
weighted avg       0.67      0.66      0.66      5951

[[1711  787  457]
 [ 556  930    9]
 [ 114   93 1294]]


# Experiment 1 - Bidirectional LSTM

In [37]:
import torch.nn as nn

class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=13,
        hidden_size=64,
        num_layers=2,
        num_classes=3,
        dropout=0.3
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):

        _, (hidden, _) = self.lstm(x)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        x = torch.cat((forward_hidden, backward_hidden), dim=1)

        x = self.dropout(x)

        return self.fc(x)

In [38]:
model = BiLSTMClassifier().to(device)

criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [39]:
import copy

num_epochs = 20
patience = 3

best_model = None
best_f1 = 0.0
patience_counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1, _, _ = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    test_f1s.append(test_f1)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f} | "
        f"Macro F1: {test_f1:.4f}"
    )

    if test_f1 > best_f1:

        best_f1 = test_f1
        patience_counter = 0

        best_model = copy.deepcopy(model.state_dict())


    else:

        patience_counter += 1

    if patience_counter >= patience:

        print("\nEarly stopping!")
        break

Epoch 01/20 | Train Loss: 0.8480 | Train Acc: 0.5783 | Test Loss: 0.8467 | Test Acc: 0.5418 | Macro F1: 0.5638
Epoch 02/20 | Train Loss: 0.6049 | Train Acc: 0.7073 | Test Loss: 0.7507 | Test Acc: 0.6207 | Macro F1: 0.6349
Epoch 03/20 | Train Loss: 0.5180 | Train Acc: 0.7638 | Test Loss: 0.7213 | Test Acc: 0.6491 | Macro F1: 0.6624
Epoch 04/20 | Train Loss: 0.4651 | Train Acc: 0.7967 | Test Loss: 0.8335 | Test Acc: 0.6495 | Macro F1: 0.6503
Epoch 05/20 | Train Loss: 0.4274 | Train Acc: 0.8135 | Test Loss: 0.8781 | Test Acc: 0.6036 | Macro F1: 0.6202
Epoch 06/20 | Train Loss: 0.3975 | Train Acc: 0.8307 | Test Loss: 1.0166 | Test Acc: 0.5666 | Macro F1: 0.5829

Early stopping!


In [40]:
model.load_state_dict(best_model)

<All keys matched successfully>

In [41]:
test_loss, test_acc, test_f1, y_true, y_pred = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"\nFinal Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")


Final Accuracy : 0.6491
Final Macro F1 : 0.6624


In [42]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)

print(cm)

              precision    recall  f1-score   support

           0       0.72      0.54      0.62      2955
           1       0.45      0.63      0.52      1495
           2       0.82      0.88      0.85      1501

    accuracy                           0.65      5951
   macro avg       0.66      0.68      0.66      5951
weighted avg       0.68      0.65      0.65      5951

[[1603 1075  277]
 [ 532  942   21]
 [  88   95 1318]]


## Experiment 1 Results

Baseline LSTM outperformed the Bidirectional LSTM.

- Baseline Accuracy: 0.6612
- BiLSTM Accuracy: 0.6491

- Baseline Macro F1: 0.6659
- BiLSTM Macro F1: 0.6624

**Decision:** Bidirectional LSTM was not selected for the final model because it did not improve classification performance.

# Experiment 2 - Dropout Analysis

In [44]:
# Dropout Experiment (0.2)

model = LSTMClassifier(
    input_size=X_train.shape[2],
    hidden_size=64,
    num_layers=2,
    num_classes=len(np.unique(y)),
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [46]:
from sklearn.metrics import accuracy_score, f1_score

In [47]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [48]:
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [49]:
num_epochs = 20

best_f1 = 0
best_model = copy.deepcopy(model.state_dict())
patience = 3
counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

train_f1s = []
test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1 = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    train_f1s.append(train_f1)
    test_f1s.append(test_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

    if test_f1 > best_f1:
        best_f1 = test_f1
        best_model = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping!")
        break

Epoch 1/20
Train Loss: 0.6671 | Acc: 0.6833 | F1: 0.6898
Test  Loss: 0.8085 | Acc: 0.5658 | F1: 0.5823
Epoch 2/20
Train Loss: 0.6168 | Acc: 0.7042 | F1: 0.7138
Test  Loss: 0.8087 | Acc: 0.5693 | F1: 0.5863
Epoch 3/20
Train Loss: 0.5920 | Acc: 0.7128 | F1: 0.7230
Test  Loss: 0.8074 | Acc: 0.5754 | F1: 0.5901
Epoch 4/20
Train Loss: 0.5650 | Acc: 0.7225 | F1: 0.7334
Test  Loss: 0.7672 | Acc: 0.5762 | F1: 0.6012
Epoch 5/20
Train Loss: 0.5332 | Acc: 0.7351 | F1: 0.7467
Test  Loss: 0.8530 | Acc: 0.5854 | F1: 0.6122
Epoch 6/20
Train Loss: 0.4990 | Acc: 0.7554 | F1: 0.7666
Test  Loss: 0.7135 | Acc: 0.6819 | F1: 0.6728
Epoch 7/20
Train Loss: 0.4677 | Acc: 0.7804 | F1: 0.7898
Test  Loss: 0.7194 | Acc: 0.7056 | F1: 0.6956
Epoch 8/20
Train Loss: 0.4405 | Acc: 0.7983 | F1: 0.8066
Test  Loss: 0.8147 | Acc: 0.6881 | F1: 0.6888
Epoch 9/20
Train Loss: 0.4294 | Acc: 0.8115 | F1: 0.8195
Test  Loss: 0.9480 | Acc: 0.6632 | F1: 0.6688
Epoch 10/20
Train Loss: 0.4125 | Acc: 0.8198 | F1: 0.8270
Test  Loss: 0.9

In [50]:
model.load_state_dict(best_model)

test_loss, test_acc, test_f1 = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("=" * 50)
print("Dropout = 0.2")
print(f"Final Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")
print("=" * 50)

Dropout = 0.2
Final Accuracy : 0.7056
Final Macro F1 : 0.6956


In [52]:
# Dropout Experiment (0.4)

model = LSTMClassifier(
    input_size=X_train.shape[2],
    hidden_size=64,
    num_layers=2,
    num_classes=len(np.unique(y)),
    dropout=0.4
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [53]:
num_epochs = 20

best_f1 = 0
best_model = copy.deepcopy(model.state_dict())
patience = 3
counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

train_f1s = []
test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1 = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    train_f1s.append(train_f1)
    test_f1s.append(test_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

    if test_f1 > best_f1:
        best_f1 = test_f1
        best_model = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping!")
        break

Epoch 1/20
Train Loss: 0.9272 | Acc: 0.5164 | F1: 0.5204
Test  Loss: 0.8851 | Acc: 0.5147 | F1: 0.5258
Epoch 2/20
Train Loss: 0.6961 | Acc: 0.6672 | F1: 0.6733
Test  Loss: 0.8215 | Acc: 0.5562 | F1: 0.5727
Epoch 3/20
Train Loss: 0.6259 | Acc: 0.7040 | F1: 0.7140
Test  Loss: 0.7458 | Acc: 0.6258 | F1: 0.6356
Epoch 4/20
Train Loss: 0.5857 | Acc: 0.7291 | F1: 0.7385
Test  Loss: 0.7672 | Acc: 0.6132 | F1: 0.6336
Epoch 5/20
Train Loss: 0.5641 | Acc: 0.7396 | F1: 0.7495
Test  Loss: 0.8248 | Acc: 0.6135 | F1: 0.6286
Epoch 6/20
Train Loss: 0.5343 | Acc: 0.7561 | F1: 0.7657
Test  Loss: 0.6971 | Acc: 0.6638 | F1: 0.6810
Epoch 7/20
Train Loss: 0.5045 | Acc: 0.7718 | F1: 0.7806
Test  Loss: 0.7384 | Acc: 0.6775 | F1: 0.6855
Epoch 8/20
Train Loss: 0.4695 | Acc: 0.7888 | F1: 0.7972
Test  Loss: 0.7670 | Acc: 0.7017 | F1: 0.7079
Epoch 9/20
Train Loss: 0.4603 | Acc: 0.7970 | F1: 0.8047
Test  Loss: 0.8944 | Acc: 0.6459 | F1: 0.6603
Epoch 10/20
Train Loss: 0.4347 | Acc: 0.8108 | F1: 0.8183
Test  Loss: 0.8

In [54]:
model.load_state_dict(best_model)

test_loss, test_acc, test_f1 = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("=" * 50)
print("Dropout = 0.4")
print(f"Final Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")
print("=" * 50)

Dropout = 0.4
Final Accuracy : 0.7017
Final Macro F1 : 0.7079


## Experiment 2 Results (Dropout Analysis)

Three dropout configurations were evaluated while keeping all other hyperparameters fixed.

| Dropout | Accuracy | Macro F1 |
|---------:|---------:|---------:|
| 0.3 (Baseline) | 0.6612 | 0.6659 |
| 0.2 | 0.7056 | 0.6956 |
| 0.4 | **0.7017** | **0.7079** |

Both modified dropout values outperformed the baseline model.

Although a dropout value of **0.2** achieved the highest Accuracy, a dropout value of **0.4** produced the best Macro F1-score, indicating better balanced classification performance across all three driving behavior classes.

Therefore, **Dropout = 0.4** was selected for the subsequent experiments.